# ChaoSSM — Phase 10: the canonical reservoir-computing control (random vs. trained input projection)

## Why this notebook exists

A reviewer raised a precise and correct methodological objection to the Phase 9 "fair test."
In that test we froze the recurrence but still **trained the input projection B**. In canonical
reservoir computing (Jaeger 2001; Lukosevicius & Jaeger 2009) both the recurrent weights **and**
the input weights are fixed and random; only the readout is trained. Training B lets the optimizer
map inputs directly onto the frozen system's contracting manifold — inducing generalized
synchronization that can suppress the reservoir's autonomous chaos *before the readout ever sees
it*. Under that objection, Phase 9 compared "a chaotic reservoir synchronized by a trained input
drive" against "a linear reservoir," not "chaos vs. no chaos."

This notebook settles the objection by running the **canonical RC protocol** and comparing it,
head to head, against the trained-B setup, as a controlled factor:

- **input = trained**: B (and the encoder) train, as in Phase 9. Reproduces the earlier setup.
- **input = random**: B and the encoder are frozen at random init; **only the readout trains**.
  This is the textbook reservoir-computing configuration.

If chaos still loses to a linear reservoir under the **random-input** protocol, the reviewer's
synchronization explanation is ruled out and the fair-test conclusion stands. If chaos *wins* (or
ties) under random-input where it lost under trained-input, the reviewer is right and the
fair-test claim must be withdrawn or rescoped.

## Design

Factorial over {input: trained, random} x {arm: chaos-edge, chaos-deep, linear} x {task: copy,
forecast}, plus a trained-everything GRU reference. Regimes are gamma-calibrated to a verified
target Lyapunov exponent (edge lambda~0-, deep lambda~+0.3), exactly as validated in Phase 9.
5 seeds. Frozen arms are cheap (no recurrent gradients); random-input arms are cheaper still
(only the readout trains). This is a small run.

## Pre-registered decision rule

For each task, compare each chaos reservoir against the linear reservoir **within the same input
condition**. The load-bearing comparison is the **random-input** column (canonical RC):
- chaos < linear on the chaos-favorable forecast task under random input  -> Phase 9 conclusion
  holds and is now defended against the synchronization objection.
- chaos >= linear under random input where it was < linear under trained input -> the objection is
  correct; the fair-test claim is withdrawn and the paper reports that trained input projection,
  not chaos itself, drove the earlier result.

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict, field
from itertools import product
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))
def setup_results_dir(project="ChaoSSM",phase="phase10_rc_control"):
    try:
        from google.colab import drive; drive.mount("/content/drive",force_remount=False)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"./{project}_results/{phase}"); print(f"[info] not colab ({type(e).__name__})")
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
gpu: Tesla T4
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase10_rc_control


## Configuration

In [2]:
SMOKE_TEST=False

@dataclass
class Config:
    state_dim:int=64; d_model:int=64; mlp_mult:int=2
    steps:int=4000; batch:int=64; warmup_frac:float=0.05; lr_min_frac:float=0.1
    grad_clip:float=1.0; gamma_init:float=1.0; lr:float=1e-3
    seeds:int=5
    eval_n:int=1500; eval_seed:int=12345
    copy_len:int=12; vocab:int=16
    fc_length:int=64; fc_bins:int=32
    r_cap_map:float=4.0
    lambda_target:dict=field(default_factory=lambda: dict(edge=-0.02, deep=0.30))
    lambda_tol:float=0.05; calib_steps:int=200; calib_warm:int=60
    # factorial axes
    inputs:tuple=("trained","random")   # <-- the new controlled factor
    chaos_regimes:tuple=("edge","deep")
    tasks:tuple=("copy","forecast")

CFG=Config()
if SMOKE_TEST:
    CFG.steps=40; CFG.seeds=2; CFG.eval_n=40; CFG.calib_steps=40; CFG.calib_warm=10
    print("*** SMOKE TEST ***")

def build_conditions(cfg):
    conds=[]
    for inp in cfg.inputs:
        for reg in cfg.chaos_regimes:
            for task in cfg.tasks:
                conds.append(dict(arm="chaos",regime=reg,input=inp,task=task))
        for task in cfg.tasks:
            conds.append(dict(arm="linear",regime="edge",input=inp,task=task))
    # GRU reference: fully trained (input trained by definition), both tasks
    for task in cfg.tasks:
        conds.append(dict(arm="gru",regime="edge",input="trained",task=task))
    return conds
CONDS=build_conditions(CFG)
print(f"conditions={len(CONDS)}  seeds={CFG.seeds}  total runs={len(CONDS)*CFG.seeds}")
print(json.dumps({k:v for k,v in asdict(CFG).items()},indent=2,default=str))

conditions=14  seeds=5  total runs=70
{
  "state_dim": 64,
  "d_model": 64,
  "mlp_mult": 2,
  "steps": 4000,
  "batch": 64,
  "warmup_frac": 0.05,
  "lr_min_frac": 0.1,
  "grad_clip": 1.0,
  "gamma_init": 1.0,
  "lr": 0.001,
  "seeds": 5,
  "eval_n": 1500,
  "eval_seed": 12345,
  "copy_len": 12,
  "vocab": 16,
  "fc_length": 64,
  "fc_bins": 32,
  "r_cap_map": 4.0,
  "lambda_target": {
    "edge": -0.02,
    "deep": 0.3
  },
  "lambda_tol": 0.05,
  "calib_steps": 200,
  "calib_warm": 60,
  "inputs": [
    "trained",
    "random"
  ],
  "chaos_regimes": [
    "edge",
    "deep"
  ],
  "tasks": [
    "copy",
    "forecast"
  ]
}


## 1. Cells (chaos / linear / GRU), gate always removed for the reservoir test

As in Phase 9's fair test we remove the gate, so the recurrence is
`h_t = fold(phi(h_{t-1}) + B x_t)` for the chaos arm and `tanh(W_lin h + B x_t)` for linear.
`freeze_recurrence()` freezes the map; `freeze_input()` additionally freezes B so the arm is a
canonical reservoir with a random, untrained input projection.

In [3]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)

class ChaosCell(nn.Module):
    def __init__(self,cfg,regime):
        super().__init__(); self.N=cfg.state_dim; self.regime=regime
        self.r_cap=float(cfg.r_cap_map)
        self.w_r=nn.Parameter(torch.zeros(self.N))
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        self.w_gamma=nn.Parameter(torch.tensor(float(math.log(math.expm1(cfg.gamma_init)))))
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def set_gamma(self,val):
        with torch.no_grad(): self.w_gamma.copy_(torch.tensor(float(math.log(math.expm1(max(val,1e-3))))))
    def phi(self,h):
        r=self.get_r(); u=h@self.W_mix.T+r*h*(1-h); return triangle_fold(self.get_gamma()*u)
    def step(self,h,x): return triangle_fold(self.phi(h)+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)
    def freeze_recurrence(self):
        for p_ in [self.w_r,self.W_mix,self.w_gamma]: p_.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class LinearCell(nn.Module):
    def __init__(self,cfg,**kw):
        super().__init__(); self.N=cfg.state_dim
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_lin=nn.Parameter(W*0.9)
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def step(self,h,x): return torch.tanh(h@self.W_lin.T+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): self.W_lin.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class GRUCellArm(nn.Module):
    def __init__(self,cfg,**kw): super().__init__(); self.N=cfg.state_dim; self.cell=nn.GRUCell(cfg.d_model,cfg.state_dim)
    def step(self,h,x): return self.cell(x,h)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): pass
    def freeze_input(self): pass

def make_cell(cfg,cond):
    if cond["arm"]=="gru": return GRUCellArm(cfg)
    if cond["arm"]=="linear": return LinearCell(cfg)
    return ChaosCell(cfg,cond["regime"])

## 2. gamma-calibration (verified regimes, identical to Phase 9)

In [4]:
from torch.func import jacrev
@torch.no_grad()
def map_lyapunov(cell,cfg,seed=0,steps=None,warm=None):
    steps=steps or cfg.calib_steps; warm=warm or cfg.calib_warm; N=cfg.state_dim
    dev=cell.W_mix.device
    g=torch.Generator(device=dev); g.manual_seed(seed)
    h=torch.full((N,),0.5,device=dev); Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=dev))[0]
    logsum=torch.zeros(N,device=dev)
    for t in range(steps+warm):
        def stp(z):
            r=cell.get_r(); u=z@cell.W_mix.T+r*z*(1-z); return triangle_fold(cell.get_gamma()*u)
        J=jacrev(stp)(h); h=stp(h)+0.02*torch.randn(N,generator=g,device=dev); h=torch.clamp(h,0.0,1.0)
        Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
        if t>=warm: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
    return float(torch.sort(logsum/steps,descending=True)[0][0])

def calibrate_gamma(cell,cfg,regime,seed=0):
    target=cfg.lambda_target[regime]; lo,hi=0.05,8.0; best=None; best_err=1e9
    for _ in range(18):
        mid=math.sqrt(lo*hi); cell.set_gamma(mid); lam=map_lyapunov(cell,cfg,seed=seed)
        err=abs(lam-target)
        if err<best_err: best_err=err; best=(mid,lam)
        if lam<target: lo=mid
        else: hi=mid
        if err<cfg.lambda_tol: break
    cell.set_gamma(best[0]); return best[0],best[1]

# sanity: verify regimes calibrate
for reg in CFG.chaos_regimes:
    c=ChaosCell(CFG,reg).to(DEVICE); gam,lam=calibrate_gamma(c,CFG,reg,seed=0)
    print(f"regime={reg:5s} target={CFG.lambda_target[reg]:+.2f} -> gamma={gam:.3f} measured lambda={lam:+.3f}")

regime=edge  target=-0.02 -> gamma=0.499 measured lambda=-0.040
regime=deep  target=+0.30 -> gamma=1.574 measured lambda=+0.267


## 3. Model with freeze_input control

In [5]:
class SeqModel(nn.Module):
    def __init__(self,cfg,cond,in_kind,in_dim,out_dim):
        super().__init__(); self.cfg=cfg; self.cond=cond; self.in_kind=in_kind
        if in_kind=="tokens": self.emb=nn.Embedding(in_dim,cfg.d_model)
        else: self.inp=nn.Linear(1,cfg.d_model)
        self.cell=make_cell(cfg,cond)
        self.ln=nn.LayerNorm(cfg.state_dim); self.readout=nn.Linear(cfg.state_dim,out_dim)
        self.achieved_lambda=float("nan")
        # reservoir arms (chaos, linear) freeze the recurrence; GRU trains fully
        if cond["arm"] in ("chaos","linear"): self.cell.freeze_recurrence()
        # canonical RC: also freeze the input projection AND the encoder
        if cond["input"]=="random" and cond["arm"] in ("chaos","linear"):
            self.cell.freeze_input()
            if in_kind=="tokens":
                self.emb.weight.requires_grad_(False)
            else:
                for p_ in self.inp.parameters(): p_.requires_grad_(False)
    def calibrate(self,cfg):
        if self.cond["arm"]=="chaos":
            _,self.achieved_lambda=calibrate_gamma(self.cell,cfg,self.cond["regime"],
                                                   seed=hash((self.cond["regime"],))%1000)
        return self
    def encode(self,x):
        return self.emb(x) if self.in_kind=="tokens" else self.inp(x.unsqueeze(-1))
    def forward(self,x):
        e=self.encode(x); B,T=e.shape[0],e.shape[1]
        h=self.cell.init_state(B,e.device,e.dtype); outs=[]
        for t in range(T):
            h=self.cell.step(h,e[:,t]); outs.append(self.readout(self.ln(h)))
        return torch.stack(outs,1)

## 4. Tasks (copy + forecast, identical to Phase 9)

In [6]:
V=CFG.vocab; SEP,PAD=V,V+1; VOCAB_SIZE=V+2
def make_copy(cfg,batch,device,gen):
    L=cfg.copy_len; src=torch.randint(0,V,(batch,L),device=device,generator=gen)
    inp=torch.cat([src,torch.full((batch,1),SEP,device=device),torch.full((batch,L),PAD,device=device)],1)
    tgt=torch.cat([torch.full((batch,L+1),-100,device=device),src],1)
    return inp,tgt,"tokens",VOCAB_SIZE,VOCAB_SIZE
def make_forecast(cfg,batch,device,gen):
    L=cfg.fc_length; r=3.9
    x0=0.1+0.8*torch.rand(batch,1,device=device,generator=gen); xs=[x0]
    for _ in range(L): xs.append(r*xs[-1]*(1-xs[-1]))
    X=torch.cat(xs,1); inp=X[:,:L]; nxt=X[:,1:L+1]
    tgt=torch.clamp((nxt*cfg.fc_bins).long(),0,cfg.fc_bins-1)
    return inp.float(),tgt,"scalar",1,cfg.fc_bins
TASK_FN=dict(copy=make_copy,forecast=make_forecast)
CHANCE=dict(copy=1.0/V, forecast=1.0/CFG.fc_bins)
def build_eval(cfg,task):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(cfg.eval_seed)
    return TASK_FN[task](cfg,cfg.eval_n,DEVICE,g)
@torch.no_grad()
def eval_acc(model,evalset,chunk=256):
    inp,tgt,_,_,_=evalset; model.eval(); c=tot=0.0
    for i in range(0,inp.shape[0],chunk):
        lg=model(inp[i:i+chunk]); tg=tgt[i:i+chunk]; m=tg!=-100; p=lg.argmax(-1)
        c+=((p==tg)&m).float().sum().item(); tot+=m.float().sum().item()
    model.train(); return c/max(tot,1)

## 5. Trainer (only trains requires_grad params — readout only under random-input)

In [7]:
def make_lr_lambda(cfg,total):
    warm=max(1,int(cfg.warmup_frac*total))
    def fn(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return cfg.lr_min_frac+(1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*p))
    return fn

def run_condition(cond,seed,cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    inp,tgt,in_kind,in_dim,out_dim=TASK_FN[cond["task"]](cfg,4,DEVICE,gen)
    model=SeqModel(cfg,cond,in_kind,in_dim,out_dim).to(DEVICE); model.calibrate(cfg)
    evalset=build_eval(cfg,cond["task"])
    params=[p for p in model.parameters() if p.requires_grad]
    n_train=sum(p.numel() for p in params)
    opt=torch.optim.AdamW(params,lr=cfg.lr,betas=(0.9,0.95))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(cfg,cfg.steps))
    div=False
    for step in range(cfg.steps):
        inp,tgt,_,_,_=TASK_FN[cond["task"]](cfg,cfg.batch,DEVICE,gen)
        lg=model(inp); L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1),ignore_index=-100)
        opt.zero_grad(set_to_none=True); L.backward()
        gn=torch.nn.utils.clip_grad_norm_(params,cfg.grad_clip).item(); opt.step(); sched.step()
        if not (math.isfinite(L.item()) and math.isfinite(gn)): div=True; break
    acc=eval_acc(model,evalset) if not div else float("nan")
    return dict(**cond,seed=seed,diverged=div,acc=acc,n_trainable=n_train,
                achieved_lambda=round(float(model.achieved_lambda),3))

## 6. Run the factorial

In [8]:
rows=[]; t0=time.time(); tot=len(CONDS)*CFG.seeds; done=0; first=None
for cond in CONDS:
    ct0=time.time()
    for sd in range(CFG.seeds):
        r=run_condition(cond,sd,CFG); rows.append(r); done+=1
        tag=f"{cond['arm']}/{cond['regime']}/in={cond['input']}/{cond['task']}"
        print(f"[{done}/{tot}] {tag:34s} s={sd} acc={r['acc']:.4f}{' DIV' if r['diverged'] else ''}")
    if first is None:
        first=time.time()-ct0; print(f"    [timing] first condition {first/60:.1f} min; projected ~{first*len(CONDS)/60:.1f} min")
RES=pd.DataFrame(rows); savetable(RES,"phase10_rc_control.csv")
print(f"\nwall clock: {(time.time()-t0)/60:.1f} min")

[1/70] chaos/edge/in=trained/copy         s=0 acc=0.0768
[2/70] chaos/edge/in=trained/copy         s=1 acc=0.0701
[3/70] chaos/edge/in=trained/copy         s=2 acc=0.0711
[4/70] chaos/edge/in=trained/copy         s=3 acc=0.1388
[5/70] chaos/edge/in=trained/copy         s=4 acc=0.0807
    [timing] first condition 10.2 min; projected ~142.7 min
[6/70] chaos/edge/in=trained/forecast     s=0 acc=0.8215
[7/70] chaos/edge/in=trained/forecast     s=1 acc=0.8121
[8/70] chaos/edge/in=trained/forecast     s=2 acc=0.8121
[9/70] chaos/edge/in=trained/forecast     s=3 acc=0.8567
[10/70] chaos/edge/in=trained/forecast     s=4 acc=0.8166
[11/70] chaos/deep/in=trained/copy         s=0 acc=0.0777
[12/70] chaos/deep/in=trained/copy         s=1 acc=0.0646
[13/70] chaos/deep/in=trained/copy         s=2 acc=0.0715
[14/70] chaos/deep/in=trained/copy         s=3 acc=0.0733
[15/70] chaos/deep/in=trained/copy         s=4 acc=0.0699
[16/70] chaos/deep/in=trained/forecast     s=0 acc=nan DIV
[17/70] chaos/deep/i

## 7. Analysis — chaos vs linear, within each input condition

In [9]:
def acc(**sel):
    m=RES.copy()
    for k,v in sel.items(): m=m[m[k]==v]
    a=m.acc.values; return a[np.isfinite(a)]

def sign_test_less(x,y):
    # fraction of seeds where x<y, paired by order; returns (n_less, n, p_two_sided)
    n=min(len(x),len(y)); import math as _m
    if n==0: return (0,0,float('nan'))
    d=np.array(x[:n])-np.array(y[:n]); nl=int((d<0).sum()); npos=int((d>0).sum()); k=min(nl,npos); N=nl+npos
    # two-sided sign test p
    from math import comb
    p=sum(comb(N,i) for i in range(0,k+1))/(2**N)*2 if N>0 else float('nan')
    return (nl,n,min(p,1.0))

print("Chaos-vs-linear on forecast (the chaos-favorable task), by input condition:\n")
summary=[]
for inp in CFG.inputs:
    lin=acc(arm="linear",input=inp,task="forecast")
    for reg in CFG.chaos_regimes:
        ch=acc(arm="chaos",regime=reg,input=inp,task="forecast")
        if not len(ch) or not len(lin): continue
        nl,n,p=sign_test_less(ch,lin)
        verdict = "chaos<linear" if ch.mean()<lin.mean() else "chaos>=linear"
        summary.append(dict(task="forecast",input=inp,regime=reg,
                            chaos_mean=round(float(ch.mean()),4),linear_mean=round(float(lin.mean()),4),
                            chaos_lt_linear_seeds=f"{nl}/{n}",sign_p=round(p,4),verdict=verdict))
        print(f"  input={inp:7s} chaos-{reg:4s} {ch.mean():.3f} vs linear {lin.mean():.3f}  "
              f"({nl}/{n} seeds chaos<linear, sign p={p:.3f})  -> {verdict}")
SUM=pd.DataFrame(summary); savetable(SUM,"phase10_forecast_contrasts.csv")

# GRU reference
for task in CFG.tasks:
    g=acc(arm="gru",task=task); print(f"[ref] GRU {task}: {g.mean():.3f}" if len(g) else f"[ref] GRU {task}: n/a")
# report trainable-param counts to make the protocol explicit
print("\nTrainable params by condition (shows readout-only under random input):")
print(RES.groupby(["arm","input"]).n_trainable.first().to_string())

Chaos-vs-linear on forecast (the chaos-favorable task), by input condition:

  input=trained chaos-edge 0.824 vs linear 0.876  (5/5 seeds chaos<linear, sign p=0.062)  -> chaos<linear
  input=random  chaos-edge 0.462 vs linear 0.251  (0/5 seeds chaos<linear, sign p=0.062)  -> chaos>=linear
  input=random  chaos-deep 0.251 vs linear 0.251  (2/5 seeds chaos<linear, sign p=1.000)  -> chaos<linear
saved /content/drive/My Drive/ChaoSSM/results/phase10_rc_control/phase10_forecast_contrasts.csv
[ref] GRU copy: 0.493
[ref] GRU forecast: 0.944

Trainable params by condition (shows readout-only under random input):
arm     input  
chaos   random      1298
        trained     6546
gru     trained    27410
linear  random      1298
        trained     6546


## 8. Figure — the key comparison

In [10]:
fig,axes=plt.subplots(1,2,figsize=(13,4.8))
for ax,task in zip(axes,CFG.tasks):
    labels=[]; means=[]; sds=[]; cols=[]
    for inp in CFG.inputs:
        for reg in CFG.chaos_regimes:
            a=acc(arm="chaos",regime=reg,input=inp,task=task)
            if len(a): labels.append(f"chaos-{reg}\n(in={inp})"); means.append(a.mean()); sds.append(a.std()); cols.append("tab:green" if inp=="random" else "tab:olive")
        a=acc(arm="linear",input=inp,task=task)
        if len(a): labels.append(f"linear\n(in={inp})"); means.append(a.mean()); sds.append(a.std()); cols.append("tab:blue" if inp=="random" else "steelblue")
    a=acc(arm="gru",task=task)
    if len(a): labels.append("GRU"); means.append(a.mean()); sds.append(a.std()); cols.append("tab:purple")
    x=np.arange(len(labels))
    ax.bar(x,means,yerr=sds,capsize=3,color=cols)
    ax.axhline(CHANCE[task],color="r",ls="--",lw=1,label="chance")
    ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=7.5); ax.set_title(task)
    ax.set_ylabel("held-out accuracy"); ax.grid(alpha=.3,axis="y"); ax.legend(fontsize=8)
fig.suptitle("Phase 10: canonical reservoir computing (random input) vs trained input")
fig.tight_layout(); savefig(fig,"phase10_rc_bars.png"); plt.close(fig)

saved /content/drive/My Drive/ChaoSSM/results/phase10_rc_control/phase10_rc_bars.png


## 9. Verdict

In [11]:
# does chaos still lose to linear under the CANONICAL (random-input) protocol on forecast?
rand_rows=[s for s in summary if s["input"]=="random"]
chaos_loses_random = all(s["verdict"]=="chaos<linear" for s in rand_rows) if rand_rows else None
# and did it also lose under trained input (reproducing Phase 9)?
trained_rows=[s for s in summary if s["input"]=="trained"]
chaos_loses_trained = all(s["verdict"]=="chaos<linear" for s in trained_rows) if trained_rows else None

if chaos_loses_random and chaos_loses_trained:
    decision=("OBJECTION RULED OUT. Chaos loses to the linear reservoir on the chaos-favorable "
              "forecast task under BOTH the trained-input setup (reproducing Phase 9) AND the "
              "canonical random-input reservoir protocol (frozen recurrence AND frozen random input "
              "projection, readout-only training). The reviewer's synchronization explanation --- "
              "that a trained input drive suppressed the chaos before the readout --- is therefore "
              "not the cause: chaos loses even when B is random and untrained. The fair-test "
              "conclusion stands and is now defended against the reservoir-computing objection.")
elif chaos_loses_trained and not chaos_loses_random:
    decision=("OBJECTION UPHELD. Chaos lost under trained input (Phase 9) but does NOT lose under the "
              "canonical random-input protocol. The reviewer is correct: training the input projection "
              "drove the earlier result via synchronization, not chaos being useless. The fair-test "
              "claim must be withdrawn or rescoped to 'trained-input frozen reservoirs', and the paper "
              "should report this distinction honestly.")
else:
    decision=("MIXED / INCONCLUSIVE at these settings; inspect the per-regime table. Report exactly "
              "which input condition and regime chaos does or does not lose in.")

verdict=dict(smoke_test=SMOKE_TEST,
             chaos_loses_random_input=chaos_loses_random,
             chaos_loses_trained_input=chaos_loses_trained,
             forecast_contrasts=summary, decision=decision,
             any_diverged=bool(RES.diverged.any()))
savejson(verdict,"phase10_verdict.json")
print(decision); print(); print(json.dumps(verdict,indent=2,default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase10_rc_control/phase10_verdict.json
OBJECTION UPHELD. Chaos lost under trained input (Phase 9) but does NOT lose under the canonical random-input protocol. The reviewer is correct: training the input projection drove the earlier result via synchronization, not chaos being useless. The fair-test claim must be withdrawn or rescoped to 'trained-input frozen reservoirs', and the paper should report this distinction honestly.

{
  "smoke_test": false,
  "chaos_loses_random_input": false,
  "chaos_loses_trained_input": true,
  "forecast_contrasts": [
    {
      "task": "forecast",
      "input": "trained",
      "regime": "edge",
      "chaos_mean": 0.8238,
      "linear_mean": 0.8755,
      "chaos_lt_linear_seeds": "5/5",
      "sign_p": 0.0625,
      "verdict": "chaos<linear"
    },
    {
      "task": "forecast",
      "input": "random",
      "regime": "edge",
      "chaos_mean": 0.4618,
      "linear_mean": 0.2509,
      "chaos_lt_linea